In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import lingam
from lingam.utils import make_dot
from cdt.metrics import SHD
from causal_comparator.utils import SHD_vectorized

In [ ]:
#TODO: create a notebook to show how to use data generation and the causal comparator.

In [ ]:
import pandas as pd
import numpy as np
from causal_comparator.data_generation import EdgePerturbationSimulator
from causal_comparator.discovery import CausalComparator
from causal_comparator.metrics import evaluate_binary_classification
import pandas as pd
import numpy as np

def align_matrix(B_est, p):
    """
    Un-shuffles the estimated matrix back to canonical order.
    """
    idx = np.argsort(p)
    return B_est[idx, :][:, idx]

#TODO: MOVE THE RUN SIMULATION TO THE QUICKSTART GUIDE.
def run_simulation(n_iterations=20, n1=1000, n2=100, n_nodes=10):
    """
    Refactored simulation runner. 
    Fixes the multiclass error by binarizing the Naive delta for E1 \ E2.
    """
    all_results = []
    
    for seed in range(n_iterations):
        rng = np.random.default_rng(seed)
        simulator = EdgePerturbationSimulator(n_nodes=n_nodes, rng=rng)
        
        # 1. Generate Truth (Canonical) and Shuffled Data (Blind)
        B1_true, B2_true, delta_true = simulator.create_graphs(edge_prob=0.3, n_positives=2)
        df1, p1 = simulator.simulate_data(B1_true, n1)
        df2, p2 = simulator.simulate_data(B2_true, n2)
        
        # 2. Discovery on Shuffled Data
        comparator = CausalComparator(df1, df2)
        
        # Define specific execution and transformation logic for each method
        methods_config = ["Naive", "Bootstrap", "RSBS"]
        
        for name in methods_config:
            # 3. Execute method and retrieve raw results
            if name == "Naive":
                comparator.estimate_naive()
                optimize = False
            elif name == "Bootstrap":
                comparator.estimate_bootstrap(n_sampling=100)
                optimize = True
            else: # RSBS
                comparator.estimate_rsbs(n_sampling=100, seed=seed)
                optimize = True
            
            # 4. Alignment: Map System 1 and System 2 back to canonical space
            B1_aligned = align_matrix(comparator.freq_i, p1)
            B2_aligned = align_matrix(comparator.freq_j, p2)
            
            # 5. Transform to Difference Classification (E1 \ E2)
            if not optimize:
                # For Naive, we must be strictly binary {0, 1}
                # Edge exists in B1 AND NOT in B2
                delta_scores = ((B1_aligned == 1) & (B2_aligned == 0)).astype(int)
            else:
                # For Bootstrap/RSBS, the score is the difference in selection probs
                # Ranges from -1 to 1; optimize loop in metrics.py handles this correctly
                delta_scores = B1_aligned - B2_aligned
            
            # 6. Evaluate using metrics.py
            scores = evaluate_binary_classification(
                y_true=delta_true, 
                y_scores=delta_scores, 
                optimize=optimize
            )
            
            scores.update({"method": name, "seed": seed})
            all_results.append(scores)
            
        print(f"Iteration {seed+1}/{n_iterations} complete.")
            
    return pd.DataFrame(all_results)
df = run_simulation()

Iteration 1/20 complete.
Iteration 2/20 complete.
Iteration 3/20 complete.
Iteration 4/20 complete.
Iteration 5/20 complete.
Iteration 6/20 complete.
Iteration 7/20 complete.
Iteration 8/20 complete.
Iteration 9/20 complete.
Iteration 10/20 complete.
Iteration 11/20 complete.
Iteration 12/20 complete.
Iteration 13/20 complete.
Iteration 14/20 complete.
Iteration 15/20 complete.
Iteration 16/20 complete.
Iteration 17/20 complete.
Iteration 18/20 complete.
Iteration 19/20 complete.
Iteration 20/20 complete.


In [132]:
df.head(10)

,auc_roc,aupr,best_f1,best_threshold,precision,recall,method,seed
0,1.000000,1.000000,1.000000,N/A,1.000000,1.0,Naive,0
1,1.000000,1.000000,1.000000,0.20202,1.000000,1.0,Bootstrap,0
2,1.000000,1.000000,1.000000,0.222222,1.000000,1.0,RSBS,0
3,0.994898,0.666667,0.800000,N/A,0.666667,1.0,Naive,1
4,1.000000,1.000000,1.000000,0.454545,1.000000,1.0,Bootstrap,1
5,1.000000,1.000000,1.000000,0.454545,1.000000,1.0,RSBS,1
6,0.984694,0.400000,0.571429,N/A,0.400000,1.0,Naive,2
7,1.000000,1.000000,1.000000,0.363636,1.000000,1.0,Bootstrap,2
8,1.000000,1.000000,1.000000,0.343434,1.000000,1.0,RSBS,2
9,1.000000,1.000000,1.000000,N/A,1.000000,1.0,Naive,3


In [13]:
df.to_csv("res_test_2.csv", index=None)

In [14]:
from causal_comparator.plotting import plot_ranking_metrics, plot_summary_performance, plot_classification_metrics
df = pd.read_csv('res_test_2.csv')
fig1 = plot_ranking_metrics(df, save_path="results_ranking.html")
fig2 = plot_summary_performance(df, save_path="results_summary.html")
fig3 = plot_classification_metrics(df, save_path="results_classification.html")
# 2. Show them if you are in a notebook
fig1.show()

In [15]:
fig2.show()

In [16]:
fig3.show()